# EE + Gripper + delta_chunk: 校验

读本地 LeRobot v3 产物 `data/output/agibot_shift`（`a_t = s_{t+1}` 管线）。

含：**写出数值对应 / 回算源 h5**、**EE & 夹爪图**、**delta_chunk 反算**、**边界越界补 0（无 mask）**。


In [72]:
from __future__ import annotations

import json
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
from scipy.spatial.transform import Rotation as R

REPO = Path("../..").resolve()  # phi_data repo root when cwd = phi_rds/examples
if not (REPO / "phi_data").is_dir():
    REPO = Path.cwd()
OUTPUT = REPO / "data/output/agibot_shift"
EPISODE = 0
PLOT_DIR = REPO / "data/output"
# 本产物对应源 episode（convert log: task355/ep662854）
SRC_H5 = REPO / "data/agibot_demo/sample_dataset/proprio_stats/355/662854/proprio_stats.h5"
# state 夹爪：官方仅 mm；写出时按 episode min/max → open01
print("output =", OUTPUT)
print("episode =", EPISODE)
print("src_h5 =", SRC_H5)
print("exists =", SRC_H5.is_file())

output = /Users/rookie/Desktop/EAI/phi_data/data/output/agibot_shift_ep0
episode = 0
src_h5 = /Users/rookie/Desktop/EAI/phi_data/data/agibot_demo/sample_dataset/proprio_stats/355/662854/proprio_stats.h5
exists = True


In [73]:
def load_episode(output: Path, episode: int):
    meta = pq.read_table(next((output / "meta/episodes").rglob("*.parquet"))).to_pylist()
    row = next(r for r in meta if int(r["episode_index"]) == episode)
    table = pq.read_table(
        output / "data" / f"chunk-{int(row['data/chunk_index']):03d}"
        / f"file-{int(row['data/file_index']):03d}.parquet"
    )
    ep = np.asarray(table.column("episode_index").to_pylist())
    mask = ep == episode

    def col(name: str) -> np.ndarray:
        return np.asarray(table.column(name).to_pylist(), dtype=np.float64)[mask]

    info = json.loads((output / "meta/info.json").read_text())
    fps = float(info.get("fps") or 30.0)
    return {
        "row": row,
        "fps": fps,
        "state": col("observation.state.arm.tcp_pose"),
        "action": col("action.arm.tcp_pose"),
        "state_d": col("observation.state.arm.tcp_pose.delta_chunk"),
        "action_d": col("action.arm.tcp_pose.delta_chunk"),
        "state_j": col("observation.state.arm.joint"),
        "action_j": col("action.arm.joint"),
        "state_g": col("observation.state.arm.gripper"),
        "action_g": col("action.arm.gripper"),
        "n": int(mask.sum()),
    }


def pose9_to_xyz_rpy(pose18: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """(N,18) dual-arm pose9 -> xyz (N,2,3) m, rpy (N,2,3) deg."""
    pose18 = np.asarray(pose18, dtype=np.float64)
    assert pose18.ndim == 2 and pose18.shape[1] == 18
    n = len(pose18)
    xyz = np.zeros((n, 2, 3), dtype=np.float64)
    rpy = np.zeros((n, 2, 3), dtype=np.float64)
    for arm in range(2):
        p = pose18[:, arm * 9 : (arm + 1) * 9]
        xyz[:, arm] = p[:, :3]
        a1, a2 = p[:, 3:6], p[:, 6:9]
        b1 = a1 / np.linalg.norm(a1, axis=1, keepdims=True).clip(1e-8)
        a2p = a2 - np.sum(b1 * a2, axis=1, keepdims=True) * b1
        b2 = a2p / np.linalg.norm(a2p, axis=1, keepdims=True).clip(1e-8)
        b3 = np.cross(b1, b2)
        mats = np.stack([b1, b2, b3], axis=-1)
        rpy[:, arm] = R.from_matrix(mats).as_euler("xyz", degrees=True)
    return xyz, rpy


data = load_episode(OUTPUT, EPISODE)
fps = data["fps"]
n = data["n"]
t = np.arange(n) / fps
s_xyz, s_rpy = pose9_to_xyz_rpy(data["state"])
a_xyz, a_rpy = pose9_to_xyz_rpy(data["action"])

# arm: a_t == s_{t+1}?
s_next_j = np.concatenate([data["state_j"][1:], data["state_j"][-1:]], axis=0)
s_next_p = np.concatenate([data["state"][1:], data["state"][-1:]], axis=0)
print(f"frames={n} fps={fps}")
print(f"joint |a_t-s_{{t+1}}| max={np.abs(data['action_j']-s_next_j).max():.3e}  allclose={np.allclose(data['action_j'], s_next_j)}")
print(f"pose  |a_t-s_{{t+1}}| max={np.abs(data['action']-s_next_p).max():.3e}  allclose={np.allclose(data['action'], s_next_p)}")
print(f"pose  |a_t-s_t|      max={np.abs(data['action']-data['state']).max():.3e}")
print(f"delta state shape={data['state_d'].shape}  action shape={data['action_d'].shape}")

frames=2418 fps=30.0
joint |a_t-s_{t+1}| max=0.000e+00  allclose=True
pose  |a_t-s_{t+1}| max=0.000e+00  allclose=True
pose  |a_t-s_t|      max=5.462e-02
delta state shape=(2418, 10, 18)  action shape=(2418, 64, 18)


## 数值对应 + 回算

**正向（写出内部一致性）**

- 臂 joint / tcp_pose：`a_t == s_{t+1}`（末帧重复）
- 夹爪：`a_t ≠ s_{t+1}`（equal_frac≈0）

**回算（写出 → 源 h5）**

- `observation.state.arm.joint` ↔ `state/joint/position`
- `action.arm.joint[t]` ↔ `state/joint/position[t+1]`（我们改写后的语义）
- gripper state：`mm = closed - open01 * (closed - open_rest)` ↔ `state/effector/position`
- gripper action：`closedness = 1 - open01` ↔ `action/effector/position`


In [74]:
def _shift(x: np.ndarray) -> np.ndarray:
    return np.concatenate([x[1:], x[-1:]], axis=0)


def _report(name: str, err: np.ndarray, atol: float) -> bool:
    mx, mean = float(np.max(err)), float(np.mean(err))
    ok = mx <= atol
    print(f"{'OK' if ok else 'FAIL'}  {name}: max={mx:.3e} mean={mean:.3e} atol={atol:g}")
    return ok


print("=" * 60)
print("A) 写出内部：a_t 是否对应 s_{t+1} / 夹爪是否不是")
print("=" * 60)
ok_all = True
ok_all &= _report("joint |a - s_{t+1}|", np.abs(data["action_j"] - _shift(data["state_j"])), 1e-6)
ok_all &= _report("pose  |a - s_{t+1}|", np.abs(data["action"] - _shift(data["state"])), 1e-5)
# a 与当前 s 应有可测差（否则等于没做 shift）
diff_st = np.abs(data["action_j"] - data["state_j"])
shifted = float(diff_st.max()) > 1e-6
ok_all &= shifted
print(f"{'OK' if shifted else 'FAIL'}  joint |a - s_t| max={diff_st.max():.3e} (期望 >0，证明已 shift)")
# gripper must NOT match next state
for side, i in [("L", 0), ("R", 1)]:
    s, a = data["state_g"][:, i], data["action_g"][:, i]
    eq1 = float((np.abs(a - _shift(s)) < 1e-4).mean())
    ok = eq1 < 0.01
    ok_all &= ok
    print(f"{'OK' if ok else 'FAIL'}  grip {side} equal_frac(a, s_{{t+1}})={eq1:.4f}  (期望≈0)")

print()
print("=" * 60)
print("B) 回算到源 h5")
print("=" * 60)
assert SRC_H5.is_file(), f"missing {SRC_H5}"
with h5py.File(SRC_H5, "r") as f:
    h5_sj = np.asarray(f["state/joint/position"], dtype=np.float64)
    h5_se = np.asarray(f["state/effector/position"], dtype=np.float64)  # mm
    h5_ae = np.asarray(f["action/effector/position"], dtype=np.float64)  # closedness
n = data["n"]
assert len(h5_sj) == n, f"len mismatch written={n} h5={len(h5_sj)}"

# joints
ok_all &= _report("state joint written vs h5", np.abs(data["state_j"] - h5_sj), 1e-5)
ok_all &= _report("action joint written vs h5 state[t+1]", np.abs(data["action_j"][:-1] - h5_sj[1:]), 1e-5)

# gripper back-calc：state 用本集 mm min/max；action 官方 closedness
lo, hi = h5_se.min(axis=0), h5_se.max(axis=0)
open_hat = (hi - h5_se) / np.maximum(hi - lo, 1e-6)
ok_all &= _report("state grip open01 vs h5 min/max affine", np.abs(data["state_g"] - open_hat), 1e-5)
closedness_hat = 1.0 - data["action_g"]
ok_all &= _report("action grip closedness_hat vs h5", np.abs(closedness_hat - h5_ae), 1e-5)
print(f"  state mm endpoints L[{lo[0]:.3f},{hi[0]:.3f}] R[{lo[1]:.3f},{hi[1]:.3f}]")

# polarity sanity: when h5 action closedness==0 (full open), written open≈1
open_cmd = h5_ae < 1e-6
if open_cmd.any():
    mean_open = float(data["action_g"][open_cmd].mean())
    ok = mean_open > 0.99
    ok_all &= ok
    print(f"{'OK' if ok else 'FAIL'}  h5 closedness=0 → written open01 mean={mean_open:.4f} (期望≈1)")

print()
print("=" * 60)
print("SUMMARY:", "ALL PASS" if ok_all else "SOME FAILED")
print("=" * 60)
assert ok_all, "numerical correspond / back-calc failed"

A) 写出内部：a_t 是否对应 s_{t+1} / 夹爪是否不是
OK  joint |a - s_{t+1}|: max=0.000e+00 mean=0.000e+00 atol=1e-06
OK  pose  |a - s_{t+1}|: max=0.000e+00 mean=0.000e+00 atol=1e-05
OK  joint |a - s_t| max=6.789e-02 (期望 >0，证明已 shift)
FAIL  grip L equal_frac(a, s_{t+1})=0.4148  (期望≈0)
FAIL  grip R equal_frac(a, s_{t+1})=0.1137  (期望≈0)

B) 回算到源 h5
OK  state joint written vs h5: max=5.960e-08 mean=1.801e-08 atol=1e-05
OK  action joint written vs h5 state[t+1]: max=5.960e-08 mean=1.802e-08 atol=1e-05
OK  state grip open01 vs h5 min/max affine: max=2.971e-08 mean=1.324e-09 atol=1e-05
OK  action grip closedness_hat vs h5: max=2.623e-08 mean=4.212e-11 atol=1e-05
  state mm endpoints L[34.933,119.800] R[34.667,119.111]
OK  h5 closedness=0 → written open01 mean=1.0000 (期望≈1)

SUMMARY: SOME FAILED


AssertionError: numerical correspond / back-calc failed

## delta_chunk 反算验证

写出公式：`Δ(t,k) = pose9( inv(T_state(t)) @ T_tgt(t ± k) )`。

- past：`T_tgt = T_state(t-k)`；future：`T_tgt = T_action(t+k)`
- **越界补 0**（无 mask / 无 pad 标记）；`info.json` 的 `pad` 只用于关节缺槽，不标 delta 时间边界

**反算（仅在未越界帧上比对）：**

`T_tgt(t ± k) = T_state(t) @ pose9_to_matrix(Δ(t,k))`

| 列 | direction | 反算出的量 |
|----|-----------|------------|
| `observation.*.delta_chunk` past | `t - k` | **state** tcp |
| `action.*.delta_chunk` future | `t + k` | **action** tcp |


In [ ]:
def pose9_from_matrix(matrix: np.ndarray) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=np.float64)
    return np.concatenate([matrix[:, :3, 3], matrix[:, :3, 0], matrix[:, :3, 1]], axis=1)


def pose9_to_matrix(pose9_values: np.ndarray) -> np.ndarray:
    pose9_values = np.asarray(pose9_values, dtype=np.float64)
    matrix = np.zeros((len(pose9_values), 4, 4), dtype=np.float64)
    col0, col1 = pose9_values[:, 3:6], pose9_values[:, 6:9]
    matrix[:, :3, 0] = col0
    matrix[:, :3, 1] = col1
    matrix[:, :3, 2] = np.cross(col0, col1)
    matrix[:, :3, 3] = pose9_values[:, 0:3]
    matrix[:, 3, 3] = 1.0
    return matrix


def _arm_mats(pose18: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    return pose9_to_matrix(pose18[:, :9]), pose9_to_matrix(pose18[:, 9:])


def invert_delta_to_tgt(
    ref18: np.ndarray,
    delta: np.ndarray,
    *,
    direction: str,
) -> list[tuple[int, np.ndarray, np.ndarray]]:
    """对每个 k：用 Δ 反算 tgt pose9，并返回与 gt 对齐后的 (k, hat18_valid, gt18_valid)。"""
    n = len(ref18)
    chunk = delta.shape[1]
    S_L, S_R = _arm_mats(ref18)
    out = []
    for k in range(chunk):
        d_L = pose9_to_matrix(delta[:, k, :9])
        d_R = pose9_to_matrix(delta[:, k, 9:])
        hat_L = S_L @ d_L
        hat_R = S_R @ d_R
        hat18 = np.concatenate([pose9_from_matrix(hat_L), pose9_from_matrix(hat_R)], axis=1)
        out.append((k, hat18))
    return out


state_d = np.asarray(data["state_d"], dtype=np.float64)
action_d = np.asarray(data["action_d"], dtype=np.float64)
state_p = np.asarray(data["state"], dtype=np.float64)
action_p = np.asarray(data["action"], dtype=np.float64)
n = data["n"]
print(f"delta shapes: past {state_d.shape}  future {action_d.shape}")

ok_all = True
atol = 1e-5

print("=" * 60)
print("A) past delta → 反算 state[t-k]")
print("=" * 60)
past_errs = []
for k, hat18 in invert_delta_to_tgt(state_p, state_d, direction="past"):
    if k == 0:
        gt18, hat_v = state_p, hat18
    else:
        # hat[t] predicts state[t-k]；有效 t>=k → hat[k:] vs state[:n-k]
        hat_v, gt18 = hat18[k:], state_p[: n - k]
    err = np.abs(hat_v - gt18)
    past_errs.append(float(err.max()))
    ok = float(err.max()) <= atol
    ok_all &= ok
    if k in (0, 1, state_d.shape[1] - 1) or not ok:
        print(f"{'OK' if ok else 'FAIL'}  k={k:2d}: max={err.max():.3e} mean={err.mean():.3e}  n={len(gt18)}")
print(f"past all-k max err = {max(past_errs):.3e}")

print()
print("=" * 60)
print("B) future delta → 反算 action[t+k]")
print("=" * 60)
fut_errs = []
for k, hat18 in invert_delta_to_tgt(state_p, action_d, direction="future"):
    if k == 0:
        gt18, hat_v = action_p, hat18
    else:
        # hat[t] predicts action[t+k]；有效 t < n-k → hat[:n-k] vs action[k:]
        hat_v, gt18 = hat18[: n - k], action_p[k:]
    err = np.abs(hat_v - gt18)
    fut_errs.append(float(err.max()))
    ok = float(err.max()) <= atol
    ok_all &= ok
    if k in (0, 1, 8, 32, action_d.shape[1] - 1) or not ok:
        print(f"{'OK' if ok else 'FAIL'}  k={k:2d}: max={err.max():.3e} mean={err.mean():.3e}  n={len(gt18)}")
print(f"future all-k max err = {max(fut_errs):.3e}")

# 可视化：用 future k=1 反算的 action xyz vs 写出 action[t+1]
k = 1
_, hat18 = invert_delta_to_tgt(state_p, action_d, direction="future")[k]
hat_v, gt18 = hat18[: n - k], action_p[k:]
t_v = t[: n - k]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
for ax, side, base in [(axes[0], "left", 0), (axes[1], "right", 9)]:
    for j, name in enumerate("xyz"):
        ax.plot(t_v, gt18[:, base + j], "-", lw=1.2, label=f"action {name} (written t+1)", color=f"C{j}")
        ax.plot(t_v, hat_v[:, base + j], "--", lw=1.0, label=f"from Δ k=1 {name}", color=f"C{j}", alpha=0.7)
    ax.set_title(f"invert future Δ → action[t+1] · {side}")
    ax.set_xlabel("t (s)")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc="best")
axes[0].set_ylabel("m")
fig.tight_layout()
dp = PLOT_DIR / "delta_chunk_invert_ep0.png"
fig.savefig(dp, dpi=140)
print("saved", dp)
plt.show()

print()
print("SUMMARY delta invert:", "ALL PASS" if ok_all else "SOME FAILED")
assert ok_all, "delta_chunk invert verification failed"

## delta_chunk 边界：越界补 0（无 mask）

实现：`shift_series` 越界填全零 pose9（不是 identity）。

| direction | 越界条件 | 写出值 |
|-----------|----------|--------|
| past `k` | `t < k`（看 `t-k`） | `Δ[t,k,:] = 0` |
| future `k` | `t ≥ N-k`（看 `t+k`） | `Δ[t,k,:] = 0` |

无单独 mask 列；训练忽略边界需自算 `frame_index` / `chunk_size`。真恒等 Δ（`k=0` past）是单位旋转的 pose9，**不是**全零。

In [ ]:
def oob_absmax(delta: np.ndarray, *, direction: str) -> list[tuple[int, float, int]]:
    """每个 k 的越界块 |Δ|max 与样本数。"""
    n, chunk, _ = delta.shape
    rows = []
    for k in range(chunk):
        if k == 0:
            rows.append((k, float("nan"), 0))  # k=0 无时间越界
            continue
        block = delta[:k, k] if direction == "past" else delta[n - k :, k]
        mx = float(np.abs(block).max()) if block.size else 0.0
        rows.append((k, mx, int(block.shape[0])))
    return rows


past_oob = oob_absmax(state_d, direction="past")
fut_oob = oob_absmax(action_d, direction="future")

ok_pad = True
print("=" * 60)
print("A) past OOB: frames t < k  → expect absmax == 0")
print("=" * 60)
for k, mx, cnt in past_oob:
    if k == 0:
        continue
    ok = mx == 0.0
    ok_pad &= ok
    if k in (1, 2, 5, state_d.shape[1] - 1) or not ok:
        print(f"{'OK' if ok else 'FAIL'}  past k={k:2d}: OOB absmax={mx:.3e}  n_oob={cnt}")

print()
print("=" * 60)
print("B) future OOB: frames t >= N-k  → expect absmax == 0")
print("=" * 60)
for k, mx, cnt in fut_oob:
    if k == 0:
        continue
    ok = mx == 0.0
    ok_pad &= ok
    if k in (1, 8, 32, action_d.shape[1] - 1) or not ok:
        print(f"{'OK' if ok else 'FAIL'}  fut  k={k:2d}: OOB absmax={mx:.3e}  n_oob={cnt}")

# 对照：中间帧非越界不应全零（pose9 常含 ~1 的旋转分量）
mid = n // 2
print()
print(f"对照 mid t={mid}: past k=1 absmax={np.abs(state_d[mid, 1]).max():.3e}  "
      f"fut k=1 absmax={np.abs(action_d[mid, 1]).max():.3e}")
print(f"对照 past k=0 (恒等 Δ) first-frame absmax={np.abs(state_d[0, 0]).max():.3e}  (≠0)")

# 热图：每帧每个 k 的 ||Δ||（L2），边界应是黑带
past_nrm = np.linalg.norm(state_d, axis=-1)   # (N, 10)
fut_nrm = np.linalg.norm(action_d, axis=-1)   # (N, 64)

fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=False)
im0 = axes[0].imshow(past_nrm.T, aspect="auto", origin="lower",
                     extent=[0, n / fps, -0.5, past_nrm.shape[1] - 0.5],
                     cmap="magma", interpolation="nearest")
axes[0].set_ylabel("past k")
axes[0].set_title("||d_past||_2  (black triangle = OOB zeros, t<k)")
fig.colorbar(im0, ax=axes[0], fraction=0.02, pad=0.02)

# future: head+tail 80 frames to show end black band
edge = min(80, n)
fut_show = np.concatenate([fut_nrm[:edge], fut_nrm[-edge:]], axis=0)
im1 = axes[1].imshow(fut_show.T, aspect="auto", origin="lower",
                     extent=[0, 2 * edge / fps, -0.5, fut_nrm.shape[1] - 0.5],
                     cmap="magma", interpolation="nearest")
axes[1].axvline(edge / fps, color="cyan", lw=0.8, alpha=0.8)
axes[1].set_ylabel("future k")
axes[1].set_xlabel("concat: head 80 frames | tail 80 frames (s)")
axes[1].set_title("||d_future||_2  (tail black = OOB zeros, t>=N-k; cyan=splice)")
fig.colorbar(im1, ax=axes[1], fraction=0.02, pad=0.02)
fig.tight_layout()
dp = PLOT_DIR / "delta_chunk_oob_zeros_ep0.png"
fig.savefig(dp, dpi=140)
print("saved", dp)
plt.show()

print()
print("SUMMARY OOB zero-pad:", "ALL PASS" if ok_pad else "SOME FAILED")
assert ok_pad, "delta_chunk OOB slots are not all zeros"


## EE xyz + rpy：state 实线 / action 虚线

臂已写成 `a_t = s_{t+1}`，虚线相对实线应有 **1 帧**超前（肉眼几乎贴合，但 `|a-s_t|` 非零）。


In [ ]:
CHANNELS = [
    ("x (m)", s_xyz, a_xyz, 0),
    ("y (m)", s_xyz, a_xyz, 1),
    ("z (m)", s_xyz, a_xyz, 2),
    ("roll (deg)", s_rpy, a_rpy, 0),
    ("pitch (deg)", s_rpy, a_rpy, 1),
    ("yaw (deg)", s_rpy, a_rpy, 2),
]
for arm, name in enumerate(["left", "right"]):
    fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)
    fig.suptitle(f"EE state vs action (a_t=s_{{t+1}}) — {name}  ep{EPISODE}")
    for ax, (label, s_arr, a_arr, i) in zip(axes.flat, CHANNELS):
        ax.plot(t, s_arr[:, arm, i], "-", lw=1.2, label="state", color="C0")
        ax.plot(t, a_arr[:, arm, i], "--", lw=1.0, label="action", color="C1", alpha=0.85)
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.3)
        if ax is axes[0, 0]:
            ax.legend(fontsize=8)
    for ax in axes[-1]:
        ax.set_xlabel("t (s)")
    fig.tight_layout()
    out = PLOT_DIR / f"ee_shift_{name}_ep0.png"
    fig.savefig(out, dpi=140)
    print("saved", out)
    plt.show()

## 夹爪：处理后的 state vs action

只画写出结果里的两条：

- **state** `observation.state.arm.gripper`（open01）
- **action** `action.arm.gripper`（open01，VR 指令映射后）

夹爪**不是** `a_t = s_{t+1}`（见上方数值检验）。

In [ ]:
print("gripper equal_frac(a, s_{t+1}):", {
    side: float((np.abs(data["action_g"][:, i] - np.concatenate([data["state_g"][1:, i], data["state_g"][-1:, i]])) < 1e-4).mean())
    for side, i in [("left", 0), ("right", 1)]
})

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
fig.suptitle("Gripper open01 (written): state vs action")
for ax, side, i in [(axes[0], "left", 0), (axes[1], "right", 1)]:
    ax.plot(t, data["state_g"][:, i], "-", lw=1.2, label="state", color="C0")
    ax.plot(t, data["action_g"][:, i], "--", lw=1.0, label="action", color="C1", alpha=0.9)
    ax.set_ylabel(f"{side} open01")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
axes[-1].set_xlabel("t (s)")
fig.tight_layout()
gp = PLOT_DIR / "grip_state_action_ep0.png"
fig.savefig(gp, dpi=140)
print("saved", gp)
plt.show()

## 可选：官方 `lerobot-dataset-viz`（Rerun）

文档：[lerobot-dataset-viz](https://huggingface-lerobot.mintlify.app/api/scripts/dataset-viz)


In [ ]:
# 按需取消注释跑 Rerun；默认跳过（.rrd 很大）
print("skip lerobot-dataset-viz (plots above are the verification)")